# 01 Generate Synthetic Training Data

This notebook creates local supervised fine-tuning data for the finance, legal, and healthcare adapters.

```mermaid
flowchart LR
    A[Domain templates] --> B[Synthetic chat records]
    B --> C[training_data/finance.json]
    B --> D[training_data/legal.json]
    B --> E[training_data/healthcare.json]
```

The data files are intentionally stored in `training_data/` to avoid shadowing Hugging Face's `datasets` Python package.

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Generate training data

Expected output: three JSON files under `training_data/`, one per adapter.

In [ ]:
from training.generate_synthetic import generate_domain, write_json

records_per_domain = int(os.getenv("SYNTHETIC_RECORDS_PER_DOMAIN", "60"))
data_dir = Path(cfg["data_dir"])
ensure_dirs(data_dir)

for adapter in settings_cfg.adapters:
    rows = generate_domain(adapter, records_per_domain)
    output_path = data_dir / f"{adapter}.json"
    write_json(output_path, rows)
    print(f"{adapter}: wrote {len(rows)} records to {output_path}")

## Inspect a sample

The sample should show the domain-specific system prompt and a concise assistant answer.

In [ ]:
import json

data_dir = Path(cfg["data_dir"])
for adapter in settings_cfg.adapters:
    path = data_dir / f"{adapter}.json"
    first = json.loads(path.read_text(encoding="utf-8"))[0]
    print(f"\n[{adapter}] {first['id']}")
    for message in first["messages"]:
        print(f"{message['role']}: {message['content'][:160]}")